In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

os.chdir(r"C:\Users\imagatsya\Downloads\credit-risk\credit-risk-model")

df = pd.read_csv("outputs/02_cleaned_data.csv", low_memory=False)
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 1,345,310
Columns: 22


In [2]:
# ── DROP REDUNDANT + UNNEEDED COLUMNS ────────
# installment correlated 0.95 with loan_amnt — redundant
# loan_status — replaced by our 'default' column

df = df.drop(columns=['installment', 'loan_status'])

print(f"Columns remaining: {df.shape[1]}")
print(df.columns.tolist())

Columns remaining: 20
['loan_amnt', 'term', 'int_rate', 'grade', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low', 'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'default']


In [3]:
# ── ENCODE CATEGORICAL COLUMNS ───────────────

cat_cols = ['grade','emp_length','home_ownership',
    'verification_status','purpose','addr_state','term']

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print("Categorical columns encoded:")
print(df[cat_cols].head())
print(f"\nData types:\n{df.dtypes}")

Categorical columns encoded:
   grade  emp_length  home_ownership  verification_status  purpose  \
0      2           2               1                    0        2   
1      2           2               1                    0       11   
2      1           2               1                    0        4   
3      5           4               1                    1        6   
4      2           5               5                    1        2   

   addr_state  term  
0          38     0  
1          41     0  
2          14     1  
3          38     1  
4          10     0  

Data types:
loan_amnt                 float64
term                        int32
int_rate                  float64
grade                       int32
emp_length                  int32
home_ownership              int32
annual_inc                float64
verification_status         int32
purpose                     int32
addr_state                  int32
dti                       float64
delinq_2yrs               float

In [4]:
# ── CREATE NEW FEATURES ───────────────────────
# Monthly payment as % of monthly income
df['payment_to_income'] = df['loan_amnt'] / (df['annual_inc'] / 12)

# Average credit score
df['credit_score_avg'] = df['fico_range_low']

# High risk flag — DTI over 35% is concerning
df['high_dti_flag'] = (df['dti'] > 35).astype(int)

print("New features created:")
print(df[['payment_to_income','credit_score_avg','high_dti_flag']].describe())

New features created:
       payment_to_income  credit_score_avg  high_dti_flag
count       1.345310e+06      1.345310e+06   1.345310e+06
mean                 inf      6.961850e+02   2.595313e-02
std                  NaN      3.185251e+01   1.589955e-01
min         2.057143e-03      6.250000e+02   0.000000e+00
25%         1.496283e+00      6.700000e+02   0.000000e+00
50%         2.400000e+00      6.900000e+02   0.000000e+00
75%         3.490909e+00      7.100000e+02   0.000000e+00
max                  inf      8.450000e+02   1.000000e+00


In [5]:
# ── FIX INFINITY VALUES ───────────────────────

median_income = df[df['annual_inc'] > 0]['annual_inc'].median()
df['annual_inc'] = df['annual_inc'].replace(0, median_income)

# Now recreate the feature safely
df['payment_to_income'] = df['loan_amnt'] / (df['annual_inc'] / 12)

# Cap extreme values at 99th percentile
cap = df['payment_to_income'].quantile(0.99)
df['payment_to_income'] = df['payment_to_income'].clip(upper=cap)

print(df['payment_to_income'].describe())
print(f"\nInf values remaining: {np.isinf(df['payment_to_income']).sum()}")

count    1.345310e+06
mean     2.559643e+00
std      1.365352e+00
min      2.057143e-03
25%      1.496000e+00
50%      2.400000e+00
75%      3.490909e+00
max      6.000000e+00
Name: payment_to_income, dtype: float64

Inf values remaining: 0


In [6]:
# TRAIN TEST SPLIT 
# Split data into training and testing sets
# Model learns from training set


X = df.drop(columns=['default'])   # everything except target
y = df['default']                   # what we predict

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training rows : {X_train.shape[0]:,}")
print(f"Testing rows  : {X_test.shape[0]:,}")
print(f"\nDefault rate train: {y_train.mean():.1%}")
print(f"Default rate test : {y_test.mean():.1%}")

Training rows : 1,076,248
Testing rows  : 269,062

Default rate train: 20.0%
Default rate test : 20.0%


In [7]:
# SAVE EVERYTHING 
X_train.to_csv("outputs/X_train.csv", index=False)
X_test.to_csv("outputs/X_test.csv", index=False)
y_train.to_csv("outputs/y_train.csv", index=False)
y_test.to_csv("outputs/y_test.csv", index=False)
df.to_csv("outputs/03_featured_data.csv", index=False)

print("✓ All files saved:")
print("  outputs/X_train.csv")
print("  outputs/X_test.csv")
print("  outputs/y_train.csv")
print("  outputs/y_test.csv")
print("  outputs/03_featured_data.csv")

✓ All files saved:
  outputs/X_train.csv
  outputs/X_test.csv
  outputs/y_train.csv
  outputs/y_test.csv
  outputs/03_featured_data.csv
